# Tuning

Find the optimal `n_clusters` and `n_segments` for a target data reduction.
Unlike tsam's built-in tuning, this evaluates each candidate across **all slices**,
so the result is optimal for your full multi-dimensional dataset.

In [ ]:
import plotly.io as pio
import xarray_plotly  # noqa: F401

import tsam_xarray
from tsam_xarray._sample_data import sample_energy_data

pio.renderers.default = "notebook_connected"

da = sample_energy_data(n_days=30)
print(f"Shape: {dict(da.sizes)}")

## Find optimal combination

Target 5% data reduction (keep 5% of original timesteps).
Tests all valid `(n_clusters, n_segments)` combinations and picks the one with lowest overall RMSE.

In [ ]:
result = tsam_xarray.find_optimal_combination(
    da,
    time_dim="time",
    cluster_dim=["variable", "region"],
    data_reduction=0.05,
    show_progress=False,
)
n_compact = result.n_clusters * result.n_segments
reduction = n_compact / da.sizes["time"]
print(f"Best: n_clusters={result.n_clusters}, n_segments={result.n_segments}")
print(f"Overall RMSE: {result.rmse:.4f}")
print(f"Data reduction: {n_compact} / {da.sizes['time']} = {reduction:.1%}")

## Summary of tested configurations

In [ ]:
result.summary

## RMSE vs data complexity

In [ ]:
import pandas as pd
import plotly.express as px

df = result.summary
df["label"] = df.apply(lambda r: f"{int(r.n_clusters)}c x {int(r.n_segments)}s", axis=1)
fig = px.scatter(
    df,
    x="timesteps",
    y="rmse",
    text="label",
    title="RMSE vs complexity (lower-left is better)",
)
fig.update_traces(textposition="top center")
fig

## Inspect the best result

The best `AggregationResult` is ready to use — it was evaluated across all slices.

In [ ]:
result.best.accuracy.rmse.to_dataframe("RMSE")

In [ ]:
result.best.cluster_representatives.sel(
    scenario="low",
    variable="solar",
).plotly.line(
    x="timestep",
    color="cluster",
    facet_col="region",
    title="Best typical periods (solar, low scenario)",
)

## With weights

Weight solar higher — the tuning picks the combination that best preserves solar patterns.

In [ ]:
result_w = tsam_xarray.find_optimal_combination(
    da,
    time_dim="time",
    cluster_dim=["variable", "region"],
    data_reduction=0.05,
    weights={"variable": {"solar": 5.0}},
    show_progress=False,
)

comparison = pd.DataFrame(
    {
        "unweighted": [result.n_clusters, result.n_segments, result.rmse],
        "solar 5x": [result_w.n_clusters, result_w.n_segments, result_w.rmse],
    },
    index=["n_clusters", "n_segments", "overall_rmse"],
)
comparison